# Feature Engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib

In [2]:
# Load the dataset
df = pd.read_csv('../data/balanced/career_multilabel_dataset_balanced.csv')

In [4]:
df.head()

,age,gender,degree_level,field_of_study,gpa,years_experience,python,java,c_cpp,sql,...,devops,networking,communication,leadership,problem_solving,teamwork,adaptability,recommended_job_1,recommended_job_2,recommended_job_3
0,31,NaN,NaN,Computer Science,2.91,4,1,0,0,1,...,0,0,2,1,4,5,4,Data Engineer,NaN,NaN
1,27,NaN,NaN,Computer Science,2.74,2,0,1,0,0,...,1,1,4,2,3,4,2,Cloud Engineer,NaN,NaN
2,29,Male,PhD,Software Engineering,3.17,1,1,0,1,0,...,0,0,2,1,4,2,2,Machine Learning Engineer,Software Engineer,Data Scientist
3,34,Male,PhD,Computer Science,3.76,4,0,0,1,1,...,1,1,5,4,5,3,4,Software Engineer,DevOps Engineer,AI Researcher
4,26,Male,Master,Cybersecurity,3.98,3,1,0,0,1,...,0,1,2,5,3,4,5,Business Analyst,Cybersecurity Analyst,AI Researcher


In [3]:
# Define the feature engineering function
def feature_engineering(df):
    df = df.copy()

    df.drop(columns=['age', 'gender', 'degree_level', 'years_experience', 
                     'recommended_job_2', 'recommended_job_3'], 
            inplace=True, errors='ignore')

    tech_skills = ['python', 'java', 'c_cpp', 'sql', 'machine_learning',
                   'data_analysis', 'cloud_computing', 'cybersecurity',
                   'web_development', 'devops', 'networking']
    soft_skills = ['communication', 'leadership', 'problem_solving', 'teamwork', 'adaptability']

    df['tech_total'] = df[tech_skills].sum(axis=1)
    df['soft_total'] = df[soft_skills].sum(axis=1)
    df['tech_soft_ratio'] = df['tech_total'] / (df['soft_total'] + 1)


    df['ds_score'] = df['machine_learning'] * 2 + df['data_analysis'] + df['sql'] + df['python']

    df['da_score'] = df['sql'] * 2 + df['data_analysis'] * 2 + df['communication']
    df['da_score_v2'] = (
        df['sql'] * 2 +
        df['data_analysis'] * 2 +
        df['python'] +
        df['communication'] +
        (1 - df['machine_learning']) +
        (1 - df['devops']) +
        (1 - df['cloud_computing'])
    )

    df['ba_score'] = (
        df['communication'] * 2 +
        df['leadership'] +
        df['problem_solving'] +
        df['data_analysis'] -
        df['python'] -
        df['machine_learning'] -
        df['devops']
    )


    df['mle_score'] = df['machine_learning'] * 2 + df['python'] + df['cloud_computing'] + df['devops']


    df['de_score'] = (
        df['sql'] * 2 +
        df['cloud_computing'] * 2 +
        df['devops'] * 2 +
        df['python'] +
        df['networking'] +
        (1 - df['machine_learning']) +
        (1 - df['data_analysis'])
    )


    df['se_score'] = (
        df['web_development'] * 2 +
        df['java'] * 2 +
        df['c_cpp'] +
        df['python'] +
        (1 - df['data_analysis']) +
        (1 - df['machine_learning'])
    )


    df['ce_score'] = (
        df['cloud_computing'] * 2 +
        df['devops'] * 2 +
        df['networking'] +
        df['cybersecurity'] +
        (1 - df['data_analysis']) -
        df['machine_learning']
    )

    
    df['is_data_engineer'] = ((df['sql'] == 1) & (df['cloud_computing'] == 1) & (df['devops'] == 1)).astype(int)
    df['is_data_analyst']  = ((df['sql'] == 1) & (df['data_analysis'] == 1) & (df['machine_learning'] == 0)).astype(int)
    df['is_software_dev']  = (((df['web_development'] == 1) | (df['java'] == 1)) & (df['data_analysis'] == 0)).astype(int)
    df['is_cloud_expert']  = ((df['cloud_computing'] == 1) & (df['devops'] == 1)).astype(int)
    
    df['is_heavy_ml']    = ((df['machine_learning'] == 1) & (df['python'] == 1)).astype(int)
    df['is_data_expert'] = ((df['sql'] == 1) & (df['data_analysis'] == 1)).astype(int)


    df['dev_vs_data'] = (df['web_development'] + df['java'] + df['c_cpp']) - (df['data_analysis'] + df['sql'])
    
    df['ml_vs_dev'] = df['machine_learning'] - (df['web_development'] + df['java'])

    df['analytics_no_infra'] = df['data_analysis'] + df['sql'] - df['devops'] - df['cloud_computing']
    df['infra_no_analytics'] = df['devops'] + df['cloud_computing'] + df['networking'] - df['data_analysis']
    df['da_vs_ds'] = df['data_analysis'] - df['machine_learning']

    return df

In [4]:
# Apply feature engineering
df = feature_engineering(df)

In [5]:
# Split the dataset into train and test sets
train,test = train_test_split(df, test_size=0.2, random_state=42, shuffle=True, stratify=df['recommended_job_1'])

In [6]:
# Prepare features and target variable
X_train = train.drop(columns=['recommended_job_1'])
y_train = train['recommended_job_1']
X_test = test.drop(columns=['recommended_job_1'])
y_test = test['recommended_job_1']

In [7]:
# Define preprocessing pipelines for numeric and categorical features
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category','str']).columns.tolist()

num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore',sparse_output=False,drop='first'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),    
    ('cat', cat_pipeline, cat_cols)
], remainder='drop', verbose_feature_names_out=False)
preprocessor.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [8]:
# Fit and transform the training data, transform the test data
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [9]:
joblib.dump(preprocessor, '../models/preprocessor_classifier.joblib')

['../models/preprocessor_classifier.joblib']